In [ ]:
from __future__ import annotations

import os
import shutil
import csv
import hashlib
import subprocess
from collections import Counter
from dataclasses import asdict, dataclass, replace
from pathlib import Path

In [ ]:
# ============================================================
# Project / paths
# ============================================================

def resolve_repository_root() -> Path:
    root = Path(
        os.environ.get(
            "PHYLOGENY_REPOSITORY_ROOT",
            Path.cwd(),
        )
    ).expanduser().resolve()

    if root.name in {"AA", "NT", "common"}:
        root = root.parent

    return root


REPOSITORY_ROOT = resolve_repository_root()
PROJECT_ROOT = REPOSITORY_ROOT / "AA"
DATA_ROOT = PROJECT_ROOT / "data"

IQTREE_EXE = Path(
    os.environ.get("IQTREE_EXE")
    or shutil.which("iqtree3")
    or "iqtree3"
)


# ============================================================
# Simulation parameters
# ============================================================

GENERATORS = ("rtree", "yule")
N_TAXA_LIST = (30,)

MEAN_BRANCH_LENGTHS = (
    0.125,
    0.250,
    0.500,
    0.625,
    0.750,
)

N_REPS = 100

SEQTYPE = "AA"
MODEL = "WAG+G5"
SEQ_LENGTH = 500

GLOBAL_SEED = 20260416

# False: keep existing outputs
# True: regenerate existing outputs
OVERWRITE = False

In [ ]:
@dataclass
class ManifestRow:
    tag: str
    generator: str
    n_taxa: int
    target_mean_branch_length: float

    seqtype: str
    model: str
    seq_length: int

    tree_seed: int
    seq_seed: int

    tree_path: str
    fasta_path: str
    log_path: str

    status: str
    message: str


def ensure_directories() -> tuple[Path, Path, Path]:
    trees_dir = DATA_ROOT / "trees"
    sim_dir = DATA_ROOT / "sim"
    manifest_dir = DATA_ROOT / "manifests"

    trees_dir.mkdir(parents=True, exist_ok=True)
    sim_dir.mkdir(parents=True, exist_ok=True)
    manifest_dir.mkdir(parents=True, exist_ok=True)

    return trees_dir, sim_dir, manifest_dir


def make_tag(
    generator: str,
    n_taxa: int,
    mean_branch_length: float,
    replicate: int,
) -> str:
    return (
        f"{generator}"
        f"_n{n_taxa}"
        f"_bl{mean_branch_length:.3f}"
        f"_rep{replicate:03d}"
    )


def stable_seed(
    global_seed: int,
    tag: str,
    purpose: str,
) -> int:
    key = f"{global_seed}:{tag}:{purpose}".encode("utf-8")
    digest = hashlib.blake2b(key, digest_size=8).digest()
    value = int.from_bytes(digest, byteorder="big")

    return value % (2**31 - 2) + 1


def load_existing_seed_map(
    manifest_path: Path,
) -> dict[str, tuple[int, int]]:
    if not manifest_path.exists():
        return {}

    seed_map: dict[str, tuple[int, int]] = {}

    with manifest_path.open(newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            tag = row.get("tag", "")
            tree_seed = row.get("tree_seed", "")
            seq_seed = row.get("seq_seed", "")

            if tag and tree_seed and seq_seed:
                seed_map[tag] = (
                    int(tree_seed),
                    int(seq_seed),
                )

    return seed_map


def build_simulation_plan(
    trees_dir: Path,
    sim_dir: Path,
    existing_seed_map: dict[str, tuple[int, int]],
) -> list[ManifestRow]:
    rows: list[ManifestRow] = []

    for generator in GENERATORS:
        for n_taxa in N_TAXA_LIST:
            for mean_bl in MEAN_BRANCH_LENGTHS:
                for rep in range(1, N_REPS + 1):
                    tag = make_tag(
                        generator=generator,
                        n_taxa=n_taxa,
                        mean_branch_length=mean_bl,
                        replicate=rep,
                    )

                    if tag in existing_seed_map:
                        tree_seed, seq_seed = existing_seed_map[tag]

                    else:
                        tree_seed = stable_seed(
                            GLOBAL_SEED,
                            tag,
                            "tree",
                        )
                        seq_seed = stable_seed(
                            GLOBAL_SEED,
                            tag,
                            "sequence",
                        )

                    tree_path = trees_dir / f"{tag}.nwk"
                    fasta_path = sim_dir / f"{tag}.fa"
                    log_path = sim_dir / f"{tag}.log"

                    rows.append(
                        ManifestRow(
                            tag=tag,
                            generator=generator,
                            n_taxa=n_taxa,
                            target_mean_branch_length=float(mean_bl),
                            seqtype=SEQTYPE,
                            model=MODEL,
                            seq_length=SEQ_LENGTH,
                            tree_seed=tree_seed,
                            seq_seed=seq_seed,
                            tree_path=str(tree_path),
                            fasta_path=str(fasta_path),
                            log_path=str(log_path),
                            status="planned",
                            message="",
                        )
                    )

    return rows


def save_manifest(
    rows: list[ManifestRow],
    output_path: Path,
    *,
    relative_to: Path | None = None,
) -> None:
    fieldnames = list(ManifestRow.__annotations__.keys())
    path_fields = {
        "tree_path",
        "fasta_path",
        "log_path",
    }

    root = (
        relative_to.expanduser().resolve()
        if relative_to is not None
        else None
    )

    with output_path.open("w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
        )
        writer.writeheader()

        for row in rows:
            record = asdict(row)

            if root is not None:
                for field in path_fields:
                    value = record.get(field)

                    if not value:
                        continue

                    path = Path(value).expanduser().resolve()

                    try:
                        record[field] = (
                            path.relative_to(root).as_posix()
                        )
                    except ValueError as error:
                        raise ValueError(
                            f"{field} is outside project root: {path}"
                        ) from error

            writer.writerow(record)


def write_tree_generation_script(
    script_path: Path,
) -> None:
    script = r'''
args <- commandArgs(trailingOnly = TRUE)
plan_csv <- args[1]

suppressPackageStartupMessages(library(ape))
suppressPackageStartupMessages(library(TreeSim))

plan <- read.csv(
    plan_csv,
    stringsAsFactors = FALSE
)

for (i in seq_len(nrow(plan))) {
    generator <- plan$generator[i]
    n_taxa <- plan$n_taxa[i]
    mean_bl <- plan$target_mean_branch_length[i]
    tree_seed <- plan$tree_seed[i]
    tree_path <- plan$tree_path[i]

    set.seed(tree_seed)

    if (generator == "rtree") {
        tree <- rtree(n = n_taxa)

        n_edges <- nrow(tree$edge)

        branch_lengths <- rexp(
            n_edges,
            rate = 1.0 / mean_bl
        )

        tree$edge.length <- pmax(
            branch_lengths,
            1e-6
        )

    } else if (generator == "yule") {
        tree <- sim.bd.taxa(
            n = n_taxa,
            numbsim = 1,
            lambda = 1,
            mu = 0
        )[[1]]

        current_mean <- mean(tree$edge.length)

        tree$edge.length <- (
            tree$edge.length
            * mean_bl
            / current_mean
        )

    } else {
        stop(
            paste(
                "Unknown generator:",
                generator
            )
        )
    }

    write.tree(
        tree,
        file = tree_path
    )
}
'''

    script_path.write_text(
        script,
        encoding="utf-8",
    )


def generate_reference_trees(
    rows: list[ManifestRow],
    manifest_dir: Path,
) -> None:
    pending_rows = [
        row
        for row in rows
        if OVERWRITE or not Path(row.tree_path).exists()
    ]

    if not pending_rows:
        print("Reference trees: all files already exist.")
        return

    plan_path = manifest_dir / "pending_tree_generation_plan.csv"
    script_path = manifest_dir / "generate_reference_trees.R"

    save_manifest(
        pending_rows,
        plan_path,
    )

    write_tree_generation_script(
        script_path,
    )

    subprocess.run(
        [
            "Rscript",
            str(script_path),
            str(plan_path),
        ],
        check=True,
    )

    print(f"Reference trees generated: {len(pending_rows)}")


def simulate_sequence(
    row: ManifestRow,
) -> ManifestRow:
    fasta_path = Path(row.fasta_path)

    if fasta_path.exists() and not OVERWRITE:
        return replace(
            row,
            status="ok",
            message="skipped_existing",
        )

    output_prefix = fasta_path.with_suffix("")

    command = [
        str(IQTREE_EXE),
        "--alisim",
        str(output_prefix),
        "--tree",
        row.tree_path,
        "--seqtype",
        row.seqtype,
        "-m",
        row.model,
        "--length",
        str(row.seq_length),
        "--seed",
        str(row.seq_seed),
        "-af",
        "fasta",
    ]

    if OVERWRITE:
        command.append("--redo")

    try:
        subprocess.run(
            command,
            check=True,
            capture_output=True,
            text=True,
        )

    except subprocess.CalledProcessError as error:
        message = (
            error.stderr
            or error.stdout
            or str(error)
        )

        return replace(
            row,
            status="failed",
            message=message[:2000],
        )

    if not fasta_path.exists():
        return replace(
            row,
            status="failed",
            message="AliSim finished, but FASTA was not found.",
        )

    return replace(
        row,
        status="ok",
        message="",
    )

In [ ]:
if not IQTREE_EXE.exists():
    raise FileNotFoundError(
        f"IQ-TREE executable was not found: {IQTREE_EXE}"
    )

trees_dir, sim_dir, manifest_dir = ensure_directories()

manifest_path = manifest_dir / "simulation_manifest.csv"

existing_seed_map = load_existing_seed_map(
    manifest_path
)

plan_rows = build_simulation_plan(
    trees_dir=trees_dir,
    sim_dir=sim_dir,
    existing_seed_map=existing_seed_map,
)

save_manifest(
    plan_rows,
    manifest_dir / "simulation_plan.csv",
)

generate_reference_trees(
    rows=plan_rows,
    manifest_dir=manifest_dir,
)

completed_rows: list[ManifestRow] = []

for index, row in enumerate(plan_rows, start=1):
    print(
        f"[{index:04d}/{len(plan_rows):04d}] "
        f"{row.tag}"
    )

    completed_rows.append(
        simulate_sequence(row)
    )

save_manifest(
    completed_rows,
    manifest_path,
    relative_to=PROJECT_ROOT,
)

status_counts = Counter(
    row.status
    for row in completed_rows
)

print()
print("Simulation completed.")

for status, count in sorted(status_counts.items()):
    print(f"{status}: {count}")

print()
print(f"Manifest: {manifest_path}")